In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, BatchNormalization, Dropout, LSTM, TimeDistributed, GlobalAveragePooling2D, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

class LipReadingModel:
    def __init__(self, input_shape, num_classes):
        self.input_shape = input_shape  # (seq_len, height, width, channels)
        self.num_classes = num_classes
    
    def build_cnn_lstm_model(self):
        """Build CNN-LSTM hybrid model"""
        frame_input = Input(shape=self.input_shape, name='frame_input')
        
        # CNN for spatial feature extraction
        x = TimeDistributed(Conv2D(64, (3, 3), activation='relu', padding='same'))(frame_input)
        x = TimeDistributed(BatchNormalization())(x)
        x = TimeDistributed(MaxPooling2D((2, 2)))(x)
        x = TimeDistributed(Dropout(0.3))(x)
        
        x = TimeDistributed(Conv2D(128, (3, 3), activation='relu', padding='same'))(x)
        x = TimeDistributed(BatchNormalization())(x)
        x = TimeDistributed(MaxPooling2D((2, 2)))(x)
        x = TimeDistributed(Dropout(0.3))(x)
        
        x = TimeDistributed(Conv2D(256, (3, 3), activation='relu', padding='same'))(x)
        x = TimeDistributed(BatchNormalization())(x)
        x = TimeDistributed(GlobalAveragePooling2D())(x)
        
        # LSTM for temporal modeling
        x = LSTM(256, return_sequences=True)(x)
        x = Dropout(0.5)(x)
        x = LSTM(256)(x)
        x = Dropout(0.5)(x)
        
        # Output layer
        outputs = Dense(self.num_classes, activation='softmax')(x)
        
        model = Model(inputs=frame_input, outputs=outputs)
        model.compile(loss='categorical_crossentropy',
                     optimizer=Adam(learning_rate=0.001),
                     metrics=['accuracy'])
        
        return model
    
    def train(self, X_train, y_train, X_val, y_val, epochs=8, batch_size=8):
        """Train the model with callbacks"""
        model = self.build_cnn_lstm_model()
        
        callbacks = [
            ModelCheckpoint('lipreading_model.h5', monitor='val_accuracy', save_best_only=True, mode='max'),
            EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
        ]
        
        history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                            epochs=epochs, batch_size=batch_size,
                            callbacks=callbacks, verbose=1)
        
        return model, history

if __name__ == "__main__":
    # Load preprocessed dataset
    X_train = np.load("X_train.npy")
    X_val = np.load("X_val.npy")
    y_train = np.load("y_train.npy")
    y_val = np.load("y_val.npy")
    vocab = np.load("vocab.npy", allow_pickle=True)

    # Train the model
    model_builder = LipReadingModel(X_train.shape[1:], len(vocab))
    model, history = model_builder.train(X_train, y_train, X_val, y_val)
    print("Model training complete!")


Epoch 1/8
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 74s/step - accuracy: 0.2855 - loss: 2.1617  

6/6 ━━━━━━━━━━━━━━━━━━━━ 530s 77s/step - accuracy: 0.2912 - loss: 2.1561 - val_accuracy: 0.0909 - val_loss: 2.6677
Epoch 2/8
6/6 ━━━━━━━━━━━━━━━━━━━━ 377s 62s/step - accuracy: 0.2572 - loss: 1.8376 - val_accuracy: 0.0909 - val_loss: 2.7090
Epoch 3/8
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 57s/step - accuracy: 0.3470 - loss: 1.5361  

6/6 ━━━━━━━━━━━━━━━━━━━━ 355s 58s/step - accuracy: 0.3439 - loss: 1.5339 - val_accuracy: 0.2727 - val_loss: 2.1950
Epoch 4/8
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 56s/step - accuracy: 0.4557 - loss: 1.2212  

6/6 ━━━━━━━━━━━━━━━━━━━━ 351s 57s/step - accuracy: 0.4604 - loss: 1.2255 - val_accuracy: 0.3636 - val_loss: 1.5233
Epoch 5/8
6/6 ━━━━━━━━━━━━━━━━━━━━ 384s 57s/step - accuracy: 0.5832 - loss: 1.1478 - val_accuracy: 0.0909 - val_loss: 2.7384
Epoch 6/8
6/6 ━━━━━━━━━━━━━━━━━━━━ 353s 58s/step - accuracy: 0.5604 - loss: 1.0705 - val_accuracy: 0.0909 - val_loss: 2.7544
Epoch 7/8
6/6 ━━━━━━━━━━━━━━━━━━━━ 339s 55s/step - accuracy: 0.7051 - loss: 0.8181 - val_accuracy: 0.0000e+00 - val_loss: 3.9583
Epoch 8/8
6/6 ━━━━━━━━━━━━━━━━━━━━ 355s 58s/step - accuracy: 0.6072 - loss: 0.8789 - val_accuracy: 0.0909 - val_loss: 3.3764
Model training complete!
